<a href="https://colab.research.google.com/github/ParthHarde/breast-cancer-project/blob/main/frontend/03_build_backend_frontend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile /content/proj/backend/main.py
"""Breast cancer ML project: educational demo API."""
import io, json
from pathlib import Path
from typing import Any, Dict

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from PIL import Image

HERE = Path(__file__).resolve().parent
MODELS = HERE / "models"
FRONTEND = HERE.parent / "frontend"
MAX_BYTES = 8 * 1024 * 1024

DISCLAIMER = ("Educational demo built from research datasets. It is not a medical device, has not been "
              "clinically validated, and must not be used to diagnose or make treatment decisions.")

NUMERIC = ["age_at_diagnosis", "tumor_size", "lymph_nodes_examined_positive", "neoplasm_histologic_grade",
           "tumor_stage", "nottingham_prognostic_index", "chemotherapy", "hormone_therapy", "radio_therapy"]

app = FastAPI(title="Breast cancer ML demo")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

meta_pipe = joblib.load(MODELS / "metabric_final_pipeline.joblib")
FEATURES = json.load(open(MODELS / "metabric_final_features.json"))["features"]
bh_model = tf.keras.models.load_model(MODELS / "breakhis_final.keras", compile=False)


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict/metabric")
def predict_metabric(payload: Dict[str, Any]):
    missing = [f for f in FEATURES if f not in payload]
    if missing:
        raise HTTPException(status_code=422, detail=f"Missing fields: {missing}")
    row = pd.DataFrame([{f: payload[f] for f in FEATURES}])
    row = row.where(pd.notna(row), np.nan)
    for c in NUMERIC:
        if c in row.columns:
            try:
                row[c] = pd.to_numeric(row[c])
            except (ValueError, TypeError):
                raise HTTPException(status_code=422, detail=f"{c} must be a number")
    try:
        p = float(meta_pipe.predict_proba(row[FEATURES])[0, 1])        # P(living)
    except Exception:
        raise HTTPException(status_code=422, detail="Could not score this input. Please check the values.")
    band = "higher risk" if p < 0.35 else ("medium" if p < 0.6 else "lower risk")
    return {"model": "Logistic Regression trained on METABRIC clinical data",
            "p_living": round(p, 3), "band": band,
            "note": "Probability that a patient in the METABRIC cohort with these values was alive at last follow-up "
                    "(deaths from other causes are included). Test ROC-AUC is about 0.70, so it is only a weak signal.",
            "disclaimer": DISCLAIMER}


@app.post("/predict/breakhis")
async def predict_breakhis(file: UploadFile = File(...)):
    data = await file.read()
    if len(data) > MAX_BYTES:
        raise HTTPException(status_code=413, detail="Image too large (max 8 MB).")
    try:
        img = Image.open(io.BytesIO(data)).convert("RGB").resize((224, 224))
    except Exception:
        raise HTTPException(status_code=400, detail="Could not read this file as an image.")
    arr = np.asarray(img, dtype="float32")[None] / 255.0
    p = float(bh_model.predict(arr, verbose=0)[0, 0])                   # P(malignant)
    return {"model": "Pre-trained CNN fine-tuned on BreakHis",
            "p_malignant": round(p, 3), "label": "malignant-like pattern" if p >= 0.5 else "benign-like pattern",
            "note": "Image-level score. Test ROC-AUC about 0.74 (95% CI 0.58 to 0.86, only 13 test patients), so treat "
                    "it as a weak signal. The model was trained on H&E breast tissue images and will be unreliable "
                    "on other stains, scanners or tissue types.",
            "disclaimer": DISCLAIMER}


if FRONTEND.exists():       # must come after the API routes
    app.mount("/", StaticFiles(directory=FRONTEND, html=True), name="frontend")

Writing /content/proj/backend/main.py


In [ ]:
import sys, json
import pandas as pd
sys.path.insert(0, "/content/proj/backend")
import main
from fastapi.testclient import TestClient
client = TestClient(main.app)
print(client.get("/health").json())

dfc = pd.read_csv(f"{M}/data/processed/metabric_clean.csv")
row = dfc.iloc[0]
payload = {f: (None if pd.isna(row[f]) else (row[f].item() if hasattr(row[f], "item") else row[f])) for f in main.FEATURES}
r = client.post("/predict/metabric", json=payload); print(r.status_code, r.json())

bad = dict(payload); bad.pop(main.FEATURES[0])
print("missing field:", client.post("/predict/metabric", json=bad).status_code)        # expect 422

with open(meta["path"].iloc[0], "rb") as f:
    r = client.post("/predict/breakhis", files={"file": ("x.png", f, "image/png")}); print(r.status_code, r.json())
print("not an image:", client.post("/predict/breakhis", files={"file": ("x.txt", b"hello", "text/plain")}).status_code)  # expect 400

{'status': 'ok'}
200 {'model': 'Logistic Regression trained on METABRIC clinical data', 'p_living': 0.143, 'band': 'higher risk', 'note': 'Probability that a patient in the METABRIC cohort with these values was alive at last follow-up (deaths from other causes are included). Test ROC-AUC is about 0.70, so it is only a weak signal.', 'disclaimer': 'Educational demo built from research datasets. It is not a medical device, has not been clinically validated, and must not be used to diagnose or make treatment decisions.'}
missing field: 422
200 {'model': 'Pre-trained CNN fine-tuned on BreakHis', 'p_malignant': 0.08, 'label': 'benign-like pattern', 'note': 'Image-level score. Test ROC-AUC about 0.74 (95% CI 0.58 to 0.86, only 13 test patients), so treat it as a weak signal. The model was trained on H&E breast tissue images and will be unreliable on other stains, scanners or tissue types.', 'disclaimer': 'Educational demo built from research datasets. It is not a medical device, has not been

In [ ]:
v = json.load(open(f"{BASE_BH}/models/versions.json"))
reqs = f"""fastapi
uvicorn[standard]
python-multipart
pillow=={v['pillow']}
numpy=={v['numpy']}
pandas=={v['pandas']}
joblib=={v['joblib']}
scikit-learn=={v['scikit-learn']}
tensorflow-cpu=={v['tensorflow']}
"""
open("/content/proj/backend/requirements.txt", "w").write(reqs)
print(reqs)

fastapi
uvicorn[standard]
python-multipart
pillow==11.3.0
numpy==2.1.3
pandas==2.2.3
joblib==1.6.0
scikit-learn==1.6.1
tensorflow-cpu==2.21.0



In [ ]:
import os
print("proj folder:", os.path.exists("/content/proj"))
print("deploy on Drive:", os.listdir(f"{BASE_BH}/deploy") if os.path.exists(f"{BASE_BH}/deploy") else "missing")
print("backup on Drive:", os.path.exists(f"{BASE_BH}/proj_backup"))

proj folder: False
deploy on Drive: ['metabric_final_pipeline.joblib', 'metabric_final_features.json', 'breakhis_final.keras']
backup on Drive: False


In [ ]:
!pip -q install fastapi "uvicorn[standard]" python-multipart httpx
import shutil
os.makedirs("/content/proj/backend/models", exist_ok=True)
os.makedirs("/content/proj/frontend", exist_ok=True)
for f in os.listdir(f"{BASE_BH}/deploy"):
    shutil.copy(f"{BASE_BH}/deploy/{f}", "/content/proj/backend/models/")
print(os.listdir("/content/proj/backend/models"))

['metabric_final_pipeline.joblib', 'metabric_final_features.json', 'breakhis_final.keras']


In [ ]:
%%writefile /content/proj/frontend/index.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Breast Cancer ML Demo</title>
<style>
  :root { --bg:#f7f8fa; --card:#fff; --ink:#1d2330; --muted:#5b6475; --line:#d9dde5; --accent:#1f5fbf; --warn:#fdf1e7; --warnline:#c24d1f; }
  @media (prefers-color-scheme: dark) { :root { --bg:#14171d; --card:#1d222b; --ink:#e8ebf0; --muted:#a2aab8; --line:#323946; --accent:#6aa5ff; --warn:#2a2018; --warnline:#e07b4a; } }
  * { box-sizing: border-box; }
  body { margin:0; background:var(--bg); color:var(--ink); font:16px/1.5 system-ui, sans-serif; }
  .banner { background:var(--warn); border-bottom:3px solid var(--warnline); padding:12px 16px; }
  main { max-width:880px; margin:0 auto; padding:16px; }
  h1 { font-size:1.5rem; margin:.5rem 0 1rem; }
  .tabs { display:flex; gap:8px; margin-bottom:12px; flex-wrap:wrap; }
  .tabs button { padding:8px 14px; border:1px solid var(--line); background:var(--card); color:var(--ink); border-radius:8px; cursor:pointer; }
  .tabs button.active { background:var(--accent); color:#fff; border-color:var(--accent); }
  section { background:var(--card); border:1px solid var(--line); border-radius:10px; padding:16px; }
  .grid { display:grid; grid-template-columns:repeat(auto-fit, minmax(240px, 1fr)); gap:12px; }
  label { display:flex; flex-direction:column; font-size:.9rem; color:var(--muted); gap:4px; }
  input, select { padding:8px; border:1px solid var(--line); border-radius:6px; background:var(--bg); color:var(--ink); font-size:1rem; }
  button.primary { margin-top:14px; padding:10px 18px; background:var(--accent); color:#fff; border:0; border-radius:8px; font-size:1rem; cursor:pointer; }
  .result { margin-top:16px; padding:12px; border:1px solid var(--line); border-radius:8px; }
  .bar { height:14px; background:var(--line); border-radius:7px; overflow:hidden; margin:8px 0; }
  .bar > div { height:100%; background:var(--accent); }
  .small { font-size:.85rem; color:var(--muted); }
  .error { color:var(--warnline); }
  #preview { max-width:100%; max-height:300px; margin-top:12px; border-radius:6px; }
  footer { margin:16px 0; }
</style>
</head>
<body>
<div class="banner" role="note"><strong>Educational demo only.</strong> This tool was built from research datasets. It is not a medical device, has not been clinically validated, and must not be used to diagnose or to make decisions about anyone's care.</div>
<main>
  <h1>Breast cancer machine learning demo</h1>
  <div class="tabs">
    <button id="tab-clin" class="active">Clinical model (METABRIC)</button>
    <button id="tab-img">Image model (BreakHis)</button>
  </div>

  <section id="clin">
    <p class="small">Enter clinical values (leave unknown ones blank). The model estimates the probability that a patient in the METABRIC cohort with these values was alive at last follow-up.</p>
    <form id="clinForm"><div id="fields" class="grid"></div><button class="primary" type="submit">Estimate</button></form>
    <div id="clinOut" class="result" hidden aria-live="polite"></div>
  </section>

  <section id="img" hidden>
    <p class="small">Upload a breast tissue microscope image (H&amp;E stained, PNG or JPG, up to 8 MB). The model gives a score for a benign-like or malignant-like pattern.</p>
    <input type="file" id="file" accept="image/*">
    <div><img id="preview" alt="Selected image preview" hidden></div>
    <button class="primary" id="go" type="button">Analyse image</button>
    <div id="imgOut" class="result" hidden aria-live="polite"></div>
  </section>

  <footer class="small">
    Models: Logistic Regression on METABRIC clinical data (test ROC-AUC about 0.70, 95% CI 0.63 to 0.77); fine-tuned MobileNetV2 on BreakHis images (test ROC-AUC 0.74, 95% CI 0.58 to 0.86, only 13 test patients).
    Datasets are research data and results do not generalise to other populations.
  </footer>
</main>

<script>
const API = "";                      // same server; change only if the API is hosted elsewhere
const $ = id => document.getElementById(id);

function show(which) {
  $("clin").hidden = which !== "clin"; $("img").hidden = which !== "img";
  $("tab-clin").classList.toggle("active", which === "clin");
  $("tab-img").classList.toggle("active", which === "img");
}
$("tab-clin").onclick = () => show("clin");
$("tab-img").onclick = () => show("img");

function addLine(box, text, cls) {
  const p = document.createElement("p"); p.textContent = text;
  if (cls) p.className = cls; box.appendChild(p);
}
function showResult(box, title, prob, lines) {
  box.hidden = false; box.replaceChildren();
  const h = document.createElement("strong"); h.textContent = title; box.appendChild(h);
  const bar = document.createElement("div"); bar.className = "bar";
  const fill = document.createElement("div"); fill.style.width = Math.round(prob * 100) + "%";
  bar.appendChild(fill); box.appendChild(bar);
  lines.forEach(([t, c]) => addLine(box, t, c));
}
function showError(box, msg) { box.hidden = false; box.replaceChildren(); addLine(box, msg, "error"); }
function nice(detail) { return typeof detail === "string" ? detail : "Something went wrong. Please check the values."; }

async function loadFields() {
  const opts = await (await fetch("options.json")).json();
  const box = $("fields");
  for (const n of opts.numeric) {
    const wrap = document.createElement("label"); wrap.append(n.label);
    let el;
    if (n.binary) {
      el = document.createElement("select");
      [["", "(not known)"], ["0", "No"], ["1", "Yes"]].forEach(([v, t]) => el.add(new Option(t, v)));
    } else {
      el = document.createElement("input"); el.type = "number"; el.step = "any";
      el.min = n.min; el.max = n.max; el.placeholder = n.min + " to " + n.max;
    }
    el.name = n.name; el.dataset.kind = "num"; wrap.appendChild(el); box.appendChild(wrap);
  }
  for (const c of opts.categorical) {
    const wrap = document.createElement("label"); wrap.append(c.label);
    const el = document.createElement("select"); el.name = c.name;
    el.add(new Option("(not known)", ""));
    c.options.forEach(o => el.add(new Option(o, o)));
    wrap.appendChild(el); box.appendChild(wrap);
  }
}

$("clinForm").addEventListener("submit", async e => {
  e.preventDefault();
  const payload = {};
  for (const el of $("clinForm").elements) {
    if (!el.name) continue;
    payload[el.name] = el.value === "" ? null : (el.dataset.kind === "num" ? Number(el.value) : el.value);
  }
  try {
    const r = await fetch(API + "/predict/metabric", { method: "POST", headers: { "Content-Type": "application/json" }, body: JSON.stringify(payload) });
    const d = await r.json();
    if (!r.ok) return showError($("clinOut"), nice(d.detail));
    showResult($("clinOut"), "Estimated probability of living: " + Math.round(d.p_living * 100) + "%  (" + d.band + ")", d.p_living,
      [[d.note, "small"], [d.disclaimer, "small"]]);
  } catch (err) { showError($("clinOut"), "Could not reach the server."); }
});

$("file").addEventListener("change", () => {
  const f = $("file").files[0];
  if (f) { $("preview").src = URL.createObjectURL(f); $("preview").hidden = false; }
});

$("go").addEventListener("click", async () => {
  const f = $("file").files[0];
  if (!f) return showError($("imgOut"), "Please choose an image first.");
  if (f.size > 8 * 1024 * 1024) return showError($("imgOut"), "Image too large (max 8 MB).");
  const fd = new FormData(); fd.append("file", f);
  try {
    const r = await fetch(API + "/predict/breakhis", { method: "POST", body: fd });
    const d = await r.json();
    if (!r.ok) return showError($("imgOut"), nice(d.detail));
    showResult($("imgOut"), "Score for malignant pattern: " + Math.round(d.p_malignant * 100) + "%  (" + d.label + ")", d.p_malignant,
      [[d.note, "small"], [d.disclaimer, "small"]]);
  } catch (err) { showError($("imgOut"), "Could not reach the server."); }
});

loadFields();
</script>
</body>
</html>

Writing /content/proj/frontend/index.html


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, shutil
import numpy as np, pandas as pd

BASE_BH = "/content/drive/MyDrive/breast-cancer-project/breakhis"
M = "/content/drive/MyDrive/breast-cancer-project/metabric"
meta = pd.read_csv(f"{BASE_BH}/data/processed/breakhis_meta.csv")
print(meta.shape)                                   # (7909, 7)
print(os.listdir(f"{BASE_BH}/deploy"))              # 3 files expected

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(7909, 7)
['metabric_final_pipeline.joblib', 'metabric_final_features.json', 'breakhis_final.keras']


In [ ]:
!pip -q install fastapi "uvicorn[standard]" python-multipart httpx
os.makedirs("/content/proj/backend/models", exist_ok=True)
os.makedirs("/content/proj/frontend", exist_ok=True)
for f in os.listdir(f"{BASE_BH}/deploy"):
    shutil.copy(f"{BASE_BH}/deploy/{f}", "/content/proj/backend/models/")
print(os.listdir("/content/proj/backend/models"))

['metabric_final_pipeline.joblib', 'metabric_final_features.json', 'breakhis_final.keras']


In [ ]:
%%writefile /content/proj/backend/main.py
"""Breast cancer ML project: educational demo API."""
import io, json
from pathlib import Path
from typing import Any, Dict

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from PIL import Image

HERE = Path(__file__).resolve().parent
MODELS = HERE / "models"
FRONTEND = HERE.parent / "frontend"
MAX_BYTES = 8 * 1024 * 1024

DISCLAIMER = ("Educational demo built from research datasets. It is not a medical device, has not been "
              "clinically validated, and must not be used to diagnose or make treatment decisions.")

NUMERIC = ["age_at_diagnosis", "tumor_size", "lymph_nodes_examined_positive", "neoplasm_histologic_grade",
           "tumor_stage", "nottingham_prognostic_index", "chemotherapy", "hormone_therapy", "radio_therapy"]

app = FastAPI(title="Breast cancer ML demo")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

meta_pipe = joblib.load(MODELS / "metabric_final_pipeline.joblib")
FEATURES = json.load(open(MODELS / "metabric_final_features.json"))["features"]
bh_model = tf.keras.models.load_model(MODELS / "breakhis_final.keras", compile=False)


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict/metabric")
def predict_metabric(payload: Dict[str, Any]):
    missing = [f for f in FEATURES if f not in payload]
    if missing:
        raise HTTPException(status_code=422, detail=f"Missing fields: {missing}")
    row = pd.DataFrame([{f: payload[f] for f in FEATURES}])
    row = row.where(pd.notna(row), np.nan)
    for c in NUMERIC:
        if c in row.columns:
            try:
                row[c] = pd.to_numeric(row[c])
            except (ValueError, TypeError):
                raise HTTPException(status_code=422, detail=f"{c} must be a number")
    try:
        p = float(meta_pipe.predict_proba(row[FEATURES])[0, 1])        # P(living)
    except Exception:
        raise HTTPException(status_code=422, detail="Could not score this input. Please check the values.")
    band = "higher risk" if p < 0.35 else ("medium" if p < 0.6 else "lower risk")
    return {"model": "Logistic Regression trained on METABRIC clinical data",
            "p_living": round(p, 3), "band": band,
            "note": "Probability that a patient in the METABRIC cohort with these values was alive at last follow-up "
                    "(deaths from other causes are included). Test ROC-AUC is about 0.70, so it is only a weak signal.",
            "disclaimer": DISCLAIMER}


@app.post("/predict/breakhis")
async def predict_breakhis(file: UploadFile = File(...)):
    data = await file.read()
    if len(data) > MAX_BYTES:
        raise HTTPException(status_code=413, detail="Image too large (max 8 MB).")
    try:
        img = Image.open(io.BytesIO(data)).convert("RGB").resize((224, 224))
    except Exception:
        raise HTTPException(status_code=400, detail="Could not read this file as an image.")
    arr = np.asarray(img, dtype="float32")[None] / 255.0
    p = float(bh_model.predict(arr, verbose=0)[0, 0])                   # P(malignant)
    return {"model": "Pre-trained CNN fine-tuned on BreakHis",
            "p_malignant": round(p, 3), "label": "malignant-like pattern" if p >= 0.5 else "benign-like pattern",
            "note": "Image-level score. Test ROC-AUC about 0.74 (95% CI 0.58 to 0.86, only 13 test patients), so treat "
                    "it as a weak signal. The model was trained on H&E breast tissue images and will be unreliable "
                    "on other stains, scanners or tissue types.",
            "disclaimer": DISCLAIMER}


if FRONTEND.exists():       # must come after the API routes
    app.mount("/", StaticFiles(directory=FRONTEND, html=True), name="frontend")

Writing /content/proj/backend/main.py


In [ ]:
import os, sys, importlib
print(os.listdir("/content/proj/backend"))        # should show: ['models', 'main.py']
print(os.path.getsize("/content/proj/backend/main.py"), "bytes")   # a few thousand

importlib.invalidate_caches()
if "/content/proj/backend" not in sys.path:
    sys.path.insert(0, "/content/proj/backend")
import main
print("main imported OK")

['main.py', 'models']
3907 bytes
main imported OK


In [ ]:
import sys
sys.path.insert(0, "/content/proj/backend")
import main
from fastapi.testclient import TestClient
client = TestClient(main.app)
print(client.get("/health").json())

dfc = pd.read_csv(f"{M}/data/processed/metabric_clean.csv")
row = dfc.iloc[0]
payload = {f: (None if pd.isna(row[f]) else (row[f].item() if hasattr(row[f], "item") else row[f])) for f in main.FEATURES}
r = client.post("/predict/metabric", json=payload); print(r.status_code, r.json())

bad = dict(payload); bad.pop(main.FEATURES[0])
print("missing field:", client.post("/predict/metabric", json=bad).status_code)        # expect 422

with open(meta["path"].iloc[0], "rb") as f:
    r = client.post("/predict/breakhis", files={"file": ("x.png", f, "image/png")}); print(r.status_code, r.json())
print("not an image:", client.post("/predict/breakhis", files={"file": ("x.txt", b"hello", "text/plain")}).status_code)  # expect 400

{'status': 'ok'}
200 {'model': 'Logistic Regression trained on METABRIC clinical data', 'p_living': 0.143, 'band': 'higher risk', 'note': 'Probability that a patient in the METABRIC cohort with these values was alive at last follow-up (deaths from other causes are included). Test ROC-AUC is about 0.70, so it is only a weak signal.', 'disclaimer': 'Educational demo built from research datasets. It is not a medical device, has not been clinically validated, and must not be used to diagnose or make treatment decisions.'}
missing field: 422
200 {'model': 'Pre-trained CNN fine-tuned on BreakHis', 'p_malignant': 0.08, 'label': 'benign-like pattern', 'note': 'Image-level score. Test ROC-AUC about 0.74 (95% CI 0.58 to 0.86, only 13 test patients), so treat it as a weak signal. The model was trained on H&E breast tissue images and will be unreliable on other stains, scanners or tissue types.', 'disclaimer': 'Educational demo built from research datasets. It is not a medical device, has not been

In [ ]:
v = json.load(open(f"{BASE_BH}/models/versions.json"))
reqs = f"""fastapi
uvicorn[standard]
python-multipart
pillow=={v['pillow']}
numpy=={v['numpy']}
pandas=={v['pandas']}
joblib=={v['joblib']}
scikit-learn=={v['scikit-learn']}
tensorflow-cpu=={v['tensorflow']}
"""
open("/content/proj/backend/requirements.txt", "w").write(reqs)
print(reqs)

fastapi
uvicorn[standard]
python-multipart
pillow==11.3.0
numpy==2.1.3
pandas==2.2.3
joblib==1.6.0
scikit-learn==1.6.1
tensorflow-cpu==2.21.0



In [ ]:
shutil.copytree("/content/proj", f"{BASE_BH}/proj_backup", dirs_exist_ok=True)
print(os.listdir(f"{BASE_BH}/proj_backup"))

['backend', 'frontend']


In [ ]:
numeric, categorical = [], []
binary = ["chemotherapy", "hormone_therapy", "radio_therapy"]
for c in main.NUMERIC:
    s = dfc[c].dropna()
    numeric.append({"name": c, "label": c.replace("_", " ").capitalize(), "binary": c in binary,
                    "min": float(s.min()), "max": float(s.max())})
for c in main.FEATURES:
    if c not in main.NUMERIC:
        categorical.append({"name": c, "label": c.replace("_", " ").capitalize(),
                            "options": sorted(map(str, dfc[c].dropna().unique()))})
json.dump({"numeric": numeric, "categorical": categorical}, open("/content/proj/frontend/options.json", "w"), indent=1)
print(len(numeric), len(categorical))      # expect 9 13

9 13


In [ ]:
%%writefile /content/proj/frontend/index.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Breast Cancer ML Demo</title>
<style>
  :root { --bg:#f7f8fa; --card:#fff; --ink:#1d2330; --muted:#5b6475; --line:#d9dde5; --accent:#1f5fbf; --warn:#fdf1e7; --warnline:#c24d1f; }
  @media (prefers-color-scheme: dark) { :root { --bg:#14171d; --card:#1d222b; --ink:#e8ebf0; --muted:#a2aab8; --line:#323946; --accent:#6aa5ff; --warn:#2a2018; --warnline:#e07b4a; } }
  * { box-sizing: border-box; }
  body { margin:0; background:var(--bg); color:var(--ink); font:16px/1.5 system-ui, sans-serif; }
  .banner { background:var(--warn); border-bottom:3px solid var(--warnline); padding:12px 16px; }
  main { max-width:880px; margin:0 auto; padding:16px; }
  h1 { font-size:1.5rem; margin:.5rem 0 1rem; }
  .tabs { display:flex; gap:8px; margin-bottom:12px; flex-wrap:wrap; }
  .tabs button { padding:8px 14px; border:1px solid var(--line); background:var(--card); color:var(--ink); border-radius:8px; cursor:pointer; }
  .tabs button.active { background:var(--accent); color:#fff; border-color:var(--accent); }
  section { background:var(--card); border:1px solid var(--line); border-radius:10px; padding:16px; }
  .grid { display:grid; grid-template-columns:repeat(auto-fit, minmax(240px, 1fr)); gap:12px; }
  label { display:flex; flex-direction:column; font-size:.9rem; color:var(--muted); gap:4px; }
  input, select { padding:8px; border:1px solid var(--line); border-radius:6px; background:var(--bg); color:var(--ink); font-size:1rem; }
  button.primary { margin-top:14px; padding:10px 18px; background:var(--accent); color:#fff; border:0; border-radius:8px; font-size:1rem; cursor:pointer; }
  .result { margin-top:16px; padding:12px; border:1px solid var(--line); border-radius:8px; }
  .bar { height:14px; background:var(--line); border-radius:7px; overflow:hidden; margin:8px 0; }
  .bar > div { height:100%; background:var(--accent); }
  .small { font-size:.85rem; color:var(--muted); }
  .error { color:var(--warnline); }
  #preview { max-width:100%; max-height:300px; margin-top:12px; border-radius:6px; }
  footer { margin:16px 0; }
</style>
</head>
<body>
<div class="banner" role="note"><strong>Educational demo only.</strong> This tool was built from research datasets. It is not a medical device, has not been clinically validated, and must not be used to diagnose or to make decisions about anyone's care.</div>
<main>
  <h1>Breast cancer machine learning demo</h1>
  <div class="tabs">
    <button id="tab-clin" class="active">Clinical model (METABRIC)</button>
    <button id="tab-img">Image model (BreakHis)</button>
  </div>

  <section id="clin">
    <p class="small">Enter clinical values (leave unknown ones blank). The model estimates the probability that a patient in the METABRIC cohort with these values was alive at last follow-up.</p>
    <form id="clinForm"><div id="fields" class="grid"></div><button class="primary" type="submit">Estimate</button></form>
    <div id="clinOut" class="result" hidden aria-live="polite"></div>
  </section>

  <section id="img" hidden>
    <p class="small">Upload a breast tissue microscope image (H&amp;E stained, PNG or JPG, up to 8 MB). The model gives a score for a benign-like or malignant-like pattern.</p>
    <input type="file" id="file" accept="image/*">
    <div><img id="preview" alt="Selected image preview" hidden></div>
    <button class="primary" id="go" type="button">Analyse image</button>
    <div id="imgOut" class="result" hidden aria-live="polite"></div>
  </section>

  <footer class="small">
    Models: Logistic Regression on METABRIC clinical data (test ROC-AUC about 0.70, 95% CI 0.63 to 0.77); fine-tuned MobileNetV2 on BreakHis images (test ROC-AUC 0.74, 95% CI 0.58 to 0.86, only 13 test patients).
    Datasets are research data and results do not generalise to other populations.
  </footer>
</main>

<script>
const API = "";                      // same server; change only if the API is hosted elsewhere
const $ = id => document.getElementById(id);

function show(which) {
  $("clin").hidden = which !== "clin"; $("img").hidden = which !== "img";
  $("tab-clin").classList.toggle("active", which === "clin");
  $("tab-img").classList.toggle("active", which === "img");
}
$("tab-clin").onclick = () => show("clin");
$("tab-img").onclick = () => show("img");

function addLine(box, text, cls) {
  const p = document.createElement("p"); p.textContent = text;
  if (cls) p.className = cls; box.appendChild(p);
}
function showResult(box, title, prob, lines) {
  box.hidden = false; box.replaceChildren();
  const h = document.createElement("strong"); h.textContent = title; box.appendChild(h);
  const bar = document.createElement("div"); bar.className = "bar";
  const fill = document.createElement("div"); fill.style.width = Math.round(prob * 100) + "%";
  bar.appendChild(fill); box.appendChild(bar);
  lines.forEach(([t, c]) => addLine(box, t, c));
}
function showError(box, msg) { box.hidden = false; box.replaceChildren(); addLine(box, msg, "error"); }
function nice(detail) { return typeof detail === "string" ? detail : "Something went wrong. Please check the values."; }

async function loadFields() {
  const opts = await (await fetch("options.json")).json();
  const box = $("fields");
  for (const n of opts.numeric) {
    const wrap = document.createElement("label"); wrap.append(n.label);
    let el;
    if (n.binary) {
      el = document.createElement("select");
      [["", "(not known)"], ["0", "No"], ["1", "Yes"]].forEach(([v, t]) => el.add(new Option(t, v)));
    } else {
      el = document.createElement("input"); el.type = "number"; el.step = "any";
      el.min = n.min; el.max = n.max; el.placeholder = n.min + " to " + n.max;
    }
    el.name = n.name; el.dataset.kind = "num"; wrap.appendChild(el); box.appendChild(wrap);
  }
  for (const c of opts.categorical) {
    const wrap = document.createElement("label"); wrap.append(c.label);
    const el = document.createElement("select"); el.name = c.name;
    el.add(new Option("(not known)", ""));
    c.options.forEach(o => el.add(new Option(o, o)));
    wrap.appendChild(el); box.appendChild(wrap);
  }
}

$("clinForm").addEventListener("submit", async e => {
  e.preventDefault();
  const payload = {};
  for (const el of $("clinForm").elements) {
    if (!el.name) continue;
    payload[el.name] = el.value === "" ? null : (el.dataset.kind === "num" ? Number(el.value) : el.value);
  }
  try {
    const r = await fetch(API + "/predict/metabric", { method: "POST", headers: { "Content-Type": "application/json" }, body: JSON.stringify(payload) });
    const d = await r.json();
    if (!r.ok) return showError($("clinOut"), nice(d.detail));
    showResult($("clinOut"), "Estimated probability of living: " + Math.round(d.p_living * 100) + "%  (" + d.band + ")", d.p_living,
      [[d.note, "small"], [d.disclaimer, "small"]]);
  } catch (err) { showError($("clinOut"), "Could not reach the server."); }
});

$("file").addEventListener("change", () => {
  const f = $("file").files[0];
  if (f) { $("preview").src = URL.createObjectURL(f); $("preview").hidden = false; }
});

$("go").addEventListener("click", async () => {
  const f = $("file").files[0];
  if (!f) return showError($("imgOut"), "Please choose an image first.");
  if (f.size > 8 * 1024 * 1024) return showError($("imgOut"), "Image too large (max 8 MB).");
  const fd = new FormData(); fd.append("file", f);
  try {
    const r = await fetch(API + "/predict/breakhis", { method: "POST", body: fd });
    const d = await r.json();
    if (!r.ok) return showError($("imgOut"), nice(d.detail));
    showResult($("imgOut"), "Score for malignant pattern: " + Math.round(d.p_malignant * 100) + "%  (" + d.label + ")", d.p_malignant,
      [[d.note, "small"], [d.disclaimer, "small"]]);
  } catch (err) { showError($("imgOut"), "Could not reach the server."); }
});

loadFields();
</script>
</body>
</html>

Overwriting /content/proj/frontend/index.html


In [ ]:
import threading, uvicorn
from google.colab import output
server = uvicorn.Server(uvicorn.Config(main.app, port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
output.serve_kernel_port_as_window(8000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
shutil.copytree("/content/proj", f"{BASE_BH}/proj_backup", dirs_exist_ok=True)
print(os.listdir(f"{BASE_BH}/proj_backup"))

['backend', 'frontend']


In [ ]:
p = "/content/proj/frontend/index.html"
s = open(p).read()
s = s.replace("</style>",
  "  @media (prefers-color-scheme: dark) { button.primary, .tabs button.active { color:#0b1220; } }\n</style>")
open(p, "w").write(s)

8070